# Assignment 2, Problem 2: Variable Elimination

In this problem you will implement the variable elimination algorithm for exact inference in a Bayesian network.

The notebook has four parts:
1. **Classes (do not edit).** `Variable`, `Factor` and `BN`.
2. **Functions to implement.** `normalize`, `restrict_factor`, `sum_out_variable`, `multiply_factors` and `VE`.
3. **Sample network (do not edit).** A small network you can use for testing.
4. **Tests.** Run these to check your code.

Use only the Python standard library. Do not modify any input factor in your functions; always create and return new ones.

## Part 1: Classes (do not edit)

In [ ]:
import math
from itertools import product


class Variable:
    """A discrete random variable with a name and a finite domain."""

    def __init__(self, name, domain):
        self.name = name
        self._domain = list(domain)
        self._evidence = None          # None means no evidence has been set

    def domain(self):
        return list(self._domain)

    def set_evidence(self, value):
        assert value in self._domain, f"{value} is not in the domain of {self.name}"
        self._evidence = value

    def clear_evidence(self):
        self._evidence = None

    def get_evidence(self):
        return self._evidence

    def __repr__(self):
        return f"Variable({self.name})"


class Factor:
    """
    A factor over an ordered list of variables (its scope).
    The values are stored in a dictionary that maps a tuple of values
    (one per variable in the scope, in scope order) to a number.
    A factor with an empty scope has the single key ().
    """

    def __init__(self, name, scope):
        self.name = name
        self.scope = list(scope)
        self.values = {}

    def get_scope(self):
        return list(self.scope)

    def set_value(self, assignment, value):
        self.values[tuple(assignment)] = value

    def get_value(self, assignment):
        return self.values[tuple(assignment)]

    def print_table(self):
        names = [v.name for v in self.scope]
        print(f"Factor {self.name}  scope = {names}")
        for assignment, value in self.values.items():
            print("  ", assignment, "->", round(value, 6))


class BN:
    """A Bayesian network: a list of variables and a list of factors (the CPTs)."""

    def __init__(self, name, variables, factors):
        self.name = name
        self._variables = list(variables)
        self._factors = list(factors)

    def variables(self):
        return list(self._variables)

    def factors(self):
        return list(self._factors)

    def clear_evidence(self):
        for v in self._variables:
            v.clear_evidence()

## Part 2: Functions to implement

Fill in the five functions below. A short description is given in each docstring.

In [ ]:
def normalize(nums):
    """
    Input: a list of numbers.
    Return a NEW list of numbers, proportional to the input, that sums to 1.
    """
    # TODO
    raise NotImplementedError


def restrict_factor(f, var, value):
    """
    Input: a factor f, a variable var in the scope of f, and a value from the domain of var.
    Return a NEW factor that is f restricted to var = value.
    The new factor's scope is f's scope without var.
    Do not modify f.
    """
    # TODO
    raise NotImplementedError


def sum_out_variable(f, var):
    """
    Input: a factor f and a variable var in the scope of f.
    Return a NEW factor obtained by summing var out of f.
    The new factor's scope is f's scope without var.
    Do not modify f.
    """
    # TODO
    raise NotImplementedError


def multiply_factors(factors):
    """
    Input: a list of factors.
    Return a NEW factor equal to the product of the factors.
    The new factor's scope is every variable that appears in any input factor
    (each variable listed once).
    Do not modify the input factors.
    """
    # TODO
    raise NotImplementedError


def VE(bayes_net, var_query, varlist_evidence):
    """
    Input: a BN, a query variable, and a list of evidence variables.
    Each evidence variable already has its value set with set_evidence().
    Return a list of probabilities P(var_query = d | evidence), one for each value d
    in var_query.domain(), in the same order as the domain.
    Do not modify any factor of bayes_net.
    """
    # TODO
    raise NotImplementedError

## Part 3: Sample network (do not edit)

A five-variable network used as a standard textbook example. Burglary (`B`) and Earthquake (`E`) can set off the Alarm (`A`), and the alarm can make John (`J`) and Mary (`M`) call. All variables are Boolean.

The helper `make_cpt` builds a factor from a table giving $P(\text{child} = \text{True} \mid \text{parents})$.

In [ ]:
def make_cpt(name, child, parents, p_true):
    """
    p_true maps a tuple of parent values to P(child = True | parents).
    For a variable with no parents use the key ().
    The factor scope is [child] + parents.
    """
    f = Factor(name, [child] + parents)
    for parent_vals, p in p_true.items():
        f.set_value((True,) + parent_vals, p)
        f.set_value((False,) + parent_vals, 1 - p)
    return f


B = Variable("Burglary", [True, False])
E = Variable("Earthquake", [True, False])
A = Variable("Alarm", [True, False])
J = Variable("JohnCalls", [True, False])
M = Variable("MaryCalls", [True, False])

F_B = make_cpt("P(B)", B, [], {(): 0.001})
F_E = make_cpt("P(E)", E, [], {(): 0.002})
F_A = make_cpt("P(A|B,E)", A, [B, E], {
    (True, True): 0.95, (True, False): 0.94,
    (False, True): 0.29, (False, False): 0.001})
F_J = make_cpt("P(J|A)", J, [A], {(True,): 0.90, (False,): 0.05})
F_M = make_cpt("P(M|A)", M, [A], {(True,): 0.70, (False,): 0.01})

alarm_net = BN("Alarm", [B, E, A, J, M], [F_B, F_E, F_A, F_J, F_M])

## Part 4: Tests

These tests check each function on small examples and then check `VE` on several queries of the sample network. A passing test is a good sign, not a proof, so add your own tests.

In [ ]:
def close(a, b, tol=1e-9):
    return math.isclose(a, b, rel_tol=tol, abs_tol=tol)


# two small variables and two factors for testing the factor operations
X = Variable("X", [0, 1])
Y = Variable("Y", [0, 1])

f = Factor("f", [X, Y])
for (x, y), val in {(0, 0): 0.1, (0, 1): 0.9, (1, 0): 0.2, (1, 1): 0.8}.items():
    f.set_value((x, y), val)

g = Factor("g", [Y])
g.set_value((0,), 0.3)
g.set_value((1,), 0.7)

# normalize
out = normalize([1, 3])
assert close(out[0], 0.25) and close(out[1], 0.75), "normalize failed"

# restrict_factor: f restricted to Y = 1 should have scope [X] with values 0.9, 0.8
r = restrict_factor(f, Y, 1)
assert r.scope == [X], "restrict_factor: wrong scope"
assert close(r.get_value((0,)), 0.9) and close(r.get_value((1,)), 0.8), "restrict_factor: wrong values"
assert len(f.values) == 4 and f.scope == [X, Y], "restrict_factor modified its input"

# sum_out_variable: summing Y out of f gives 1.0 and 1.0; summing X out gives 0.3 and 1.7
s = sum_out_variable(f, Y)
assert s.scope == [X], "sum_out_variable: wrong scope"
assert close(s.get_value((0,)), 1.0) and close(s.get_value((1,)), 1.0), "sum_out_variable: wrong values"
s2 = sum_out_variable(f, X)
assert close(s2.get_value((0,)), 0.3) and close(s2.get_value((1,)), 1.7), "sum_out_variable: wrong values"
assert len(f.values) == 4 and f.scope == [X, Y], "sum_out_variable modified its input"

# multiply_factors: scope is the union of the scopes, values are products
p = multiply_factors([f, g])
assert set(p.scope) == {X, Y}, "multiply_factors: wrong scope"
lookup = lambda x, y: p.get_value(tuple({X: x, Y: y}[v] for v in p.scope))
assert close(lookup(0, 0), 0.03) and close(lookup(0, 1), 0.63), "multiply_factors: wrong values"
assert close(lookup(1, 0), 0.06) and close(lookup(1, 1), 0.56), "multiply_factors: wrong values"
assert len(f.values) == 4 and len(g.values) == 2, "multiply_factors modified its input"

print("Factor operation tests passed.")

In [ ]:
def run_query(bn, query, evidence):
    """evidence is a dict {variable: value}. Returns the result of VE."""
    bn.clear_evidence()
    for var, val in evidence.items():
        var.set_evidence(val)
    result = VE(bn, query, list(evidence.keys()))
    bn.clear_evidence()
    return result


# (query variable, evidence, expected probabilities in domain order [True, False])
cases = [
    (B, {}, [0.001, 0.999]),
    (J, {}, [0.05214, 0.94786]),
    (A, {J: True}, [0.04344, 0.95656]),
    (E, {A: True}, [0.23101, 0.76899]),
    (E, {A: True, B: True}, [0.00202, 0.99798]),
    (B, {J: True, M: False}, [0.00513, 0.99487]),
    (M, {B: True, E: False}, [0.6586, 0.3414]),
]

all_ok = True
for query, evidence, expected in cases:
    got = run_query(alarm_net, query, evidence)
    ok = len(got) == len(expected) and all(close(a, b, 1e-4) for a, b in zip(got, expected))
    all_ok = all_ok and ok
    label = ", ".join(f"{v.name}={val}" for v, val in evidence.items()) or "no evidence"
    print(("PASS" if ok else "FAIL"), f"P({query.name} | {label}) = {[round(x, 5) for x in got]}")
assert all_ok, "At least one VE query gave the wrong answer."

# VE must not change the network's factors
before = [dict(fac.values) for fac in alarm_net.factors()]
run_query(alarm_net, B, {J: True})
after = [dict(fac.values) for fac in alarm_net.factors()]
assert before == after, "VE modified a factor of the Bayes net."
print("All tests passed.")